# Week 6 – Tuning, Tracking and Serialization
## Movie Recommendation System

Important: GridSearchCV is fitted on the Week 5 training partition only. The held-out test partition is not used for hyperparameter selection. The same `random_state=42` split is recreated to keep the Week 5 and Week 6 comparison consistent.

## 1. Imports, paths and reproducibility

In [1]:
# If needed:
# %pip install pandas numpy scikit-learn scikit-surprise mlflow joblib matplotlib

import random
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import mlflow
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from surprise import Dataset, Reader, SVD
from surprise.model_selection import GridSearchCV

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROCESSED_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")
REPORT_DIR = Path("../reports")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("MLflow version:", mlflow.__version__)
print("Processed directory:", PROCESSED_DIR.resolve())
print("Model directory:", MODEL_DIR.resolve())

MLflow version: 3.18.0
Processed directory: C:\Users\PC\Documents\GitHub\Movie-recommendation-system\Backend\data\processed
Model directory: C:\Users\PC\Documents\GitHub\Movie-recommendation-system\Backend\models


## 2. Load ratings and recreate the Week 5 split

In [2]:
ratings_path = PROCESSED_DIR / "ratings_clean.csv"
if not ratings_path.exists():
    raise FileNotFoundError(f"Cannot find {ratings_path.resolve()}")

ratings = pd.read_csv(ratings_path)
ratings = ratings[["userId", "movieId", "rating"]].dropna().copy()
ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")
ratings = ratings.dropna(subset=["rating"]).reset_index(drop=True)

train_data, test_data = train_test_split(
    ratings, test_size=0.20, random_state=SEED
)
train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)
y_test = test_data["rating"].to_numpy(dtype=float)

print("Full ratings:", ratings.shape)
print("Train:", train_data.shape)
print("Test:", test_data.shape)

Full ratings: (101701, 3)
Train: (81360, 3)
Test: (20341, 3)


## 3. Load Week 5 metrics (for comparison)

In [3]:
reg_path = PROCESSED_DIR / "regression_metrics.csv"
if not reg_path.exists():
    raise FileNotFoundError("Run the Week 5 evaluation notebook first: regression_metrics.csv is missing.")
week5_regression = pd.read_csv(reg_path)
display(week5_regression.round(4))

baseline_row = week5_regression[
    week5_regression["Model"].str.contains("Baseline", case=False, na=False)
]
if baseline_row.empty:
    raise ValueError("Baseline row not found in regression_metrics.csv")
baseline_test_rmse = float(baseline_row.iloc[0]["RMSE"])
baseline_test_mae = float(baseline_row.iloc[0]["MAE"])

,Model,MAE,RMSE
0,Baseline (Mean),0.8340,1.0394
1,SVD,0.6730,0.8691
2,NCF,0.6702,0.8800


## 4. Prepare Surprise training data ONLY

In [4]:
reader = Reader(rating_scale=(0.5, 5.0))
surprise_train_data = Dataset.load_from_df(
    train_data[["userId", "movieId", "rating"]], reader
)
print("Surprise training dataset prepared.")
print("Rows available for cross-validation:", len(train_data))
print("The held-out test set is NOT passed to GridSearchCV.")

Surprise training dataset prepared.
Rows available for cross-validation: 81360
The held-out test set is NOT passed to GridSearchCV.


## 5. Hyperparameter optimization with GridSearchCV

In [5]:
param_grid = {
    "n_factors": [50, 100],
    "n_epochs": [20, 30],
    "lr_all": [0.002, 0.005],
    "reg_all": [0.02, 0.05],
    "random_state": [SEED]
}

gs = GridSearchCV(
    SVD,
    param_grid,
    measures=["rmse", "mae"],
    cv=3,
    joblib_verbose=1
)
gs.fit(surprise_train_data)

best_params = gs.best_params["rmse"]
best_cv_rmse = float(gs.best_score["rmse"])
best_cv_mae = float(gs.best_score["mae"])

print("Best CV RMSE:", round(best_cv_rmse, 4))
print("Best CV MAE:", round(best_cv_mae, 4))
print("Best parameters:", best_params)

cv_results = pd.DataFrame(gs.cv_results)
cv_results.to_csv(PROCESSED_DIR / "svd_gridsearch_results.csv", index=False)

Best CV RMSE: 0.8685
Best CV MAE: 0.6684
Best parameters: {'n_factors': 50, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.05, 'random_state': 42}


## 6. Train the tuned SVD on the full training partition

In [6]:
best_svd = SVD(**best_params)
final_trainset = surprise_train_data.build_full_trainset()
best_svd.fit(final_trainset)
print("Tuned SVD training completed.")

Tuned SVD training completed.


## 7. Final evaluation on the untouched test set

In [7]:
tuned_predictions = np.array([
    best_svd.predict(row.userId, row.movieId).est
    for row in test_data.itertuples(index=False)
], dtype=float)

tuned_test_mae = float(mean_absolute_error(y_test, tuned_predictions))
tuned_test_rmse = float(np.sqrt(mean_squared_error(y_test, tuned_predictions)))

print(f"Tuned SVD test MAE:  {tuned_test_mae:.4f}")
print(f"Tuned SVD test RMSE: {tuned_test_rmse:.4f}")
print(f"Week 5 Baseline test MAE:  {baseline_test_mae:.4f}")
print(f"Week 5 Baseline test RMSE: {baseline_test_rmse:.4f}")

tuned_test_results = pd.DataFrame([{
    "Model": "SVD Tuned",
    "CV_MAE": best_cv_mae,
    "CV_RMSE": best_cv_rmse,
    "Test_MAE": tuned_test_mae,
    "Test_RMSE": tuned_test_rmse,
    "Baseline_Test_MAE": baseline_test_mae,
    "Baseline_Test_RMSE": baseline_test_rmse
}])
display(tuned_test_results.round(4))
tuned_test_results.to_csv(PROCESSED_DIR / "tuned_svd_test_results.csv", index=False)

Tuned SVD test MAE:  0.6619
Tuned SVD test RMSE: 0.8556
Week 5 Baseline test MAE:  0.8340
Week 5 Baseline test RMSE: 1.0394


,Model,CV_MAE,CV_RMSE,Test_MAE,Test_RMSE,Baseline_Test_MAE,Baseline_Test_RMSE
0,SVD Tuned,0.6684,0.8685,0.6619,0.8556,0.834,1.0394


## 8. Acceptance criteria and validation

The criteria below are project-level checks. The model must beat the Week 5 baseline on the held-out test set; the model file must exist and reload; and its reloaded predictions must match the in-memory model. Do not claim acceptance if a check fails.

In [8]:
acceptance_checks = {
    "Tuned model improves test RMSE over baseline": tuned_test_rmse < baseline_test_rmse,
    "Tuned model improves test MAE over baseline": tuned_test_mae < baseline_test_mae,
    "Model training completed": best_svd is not None
}

for criterion, passed in acceptance_checks.items():
    print(f"{'PASS' if passed else 'FAIL'} – {criterion}")

accepted = all(acceptance_checks.values())
print("\nOverall acceptance:", "ACCEPTED" if accepted else "NOT ACCEPTED")
pd.DataFrame([
    {"criterion": k, "passed": bool(v)} for k, v in acceptance_checks.items()
]).to_csv(PROCESSED_DIR / "acceptance_criteria_results.csv", index=False)

PASS – Tuned model improves test RMSE over baseline
PASS – Tuned model improves test MAE over baseline
PASS – Model training completed

Overall acceptance: ACCEPTED


## 9. Model serialization and reload verification

In [9]:
model_path = MODEL_DIR / "best_svd_model.pkl"
joblib.dump(best_svd, model_path)
print("Saved model:", model_path.resolve())

loaded_svd = joblib.load(model_path)
sample = test_data.iloc[0]
original_est = best_svd.predict(sample["userId"], sample["movieId"]).est
loaded_est = loaded_svd.predict(sample["userId"], sample["movieId"]).est
serialization_ok = bool(np.isclose(original_est, loaded_est, atol=1e-10))

print("Reloaded prediction:", round(loaded_est, 6))
print("Original prediction:", round(original_est, 6))
print("Serialization verification:", "PASS" if serialization_ok else "FAIL")
print("Model file exists:", model_path.exists())

Saved model: C:\Users\PC\Documents\GitHub\Movie-recommendation-system\Backend\models\best_svd_model.pkl
Reloaded prediction: 3.011956
Original prediction: 3.011956
Serialization verification: PASS
Model file exists: True


## 10. MLflow experiment tracking

In [10]:
mlflow.set_experiment("Movie_Recommendation_System")

with mlflow.start_run(run_name="SVD_Tuned_Week6"):
    mlflow.log_param("model", "SVD")
    mlflow.log_param("cv_folds", 3)
    mlflow.log_param("optimization_metric", "RMSE")
    mlflow.log_params({k: v for k, v in best_params.items()})
    mlflow.log_metric("validation_rmse", best_cv_rmse)
    mlflow.log_metric("validation_mae", best_cv_mae)
    mlflow.log_metric("test_rmse", tuned_test_rmse)
    mlflow.log_metric("test_mae", tuned_test_mae)
    mlflow.log_metric("baseline_test_rmse", baseline_test_rmse)
    mlflow.log_metric("baseline_test_mae", baseline_test_mae)
    mlflow.set_tag("project", "Movie Recommendation System")
    mlflow.set_tag("week", "Week 6")
    mlflow.set_tag("model_type", "Collaborative Filtering")
    mlflow.set_tag("acceptance", "ACCEPTED" if accepted and serialization_ok else "NOT ACCEPTED")
    mlflow.log_artifact(str(model_path), artifact_path="models")
    mlflow.log_artifact(str(PROCESSED_DIR / "tuned_svd_test_results.csv"), artifact_path="evaluation")
    mlflow.log_artifact(str(PROCESSED_DIR / "acceptance_criteria_results.csv"), artifact_path="validation")

print("MLflow run logged successfully.")
print("Tracking URI:", mlflow.get_tracking_uri())

MLflow run logged successfully.
Tracking URI: sqlite:///c%3A/Users/PC/Documents/GitHub/Movie-recommendation-system/Backend/notebooks/mlflow.db


## 11. Inspect MLflow runs

In [11]:
experiment = mlflow.get_experiment_by_name("Movie_Recommendation_System")
if experiment is None:
    print("Experiment not found.")
    runs = pd.DataFrame()
else:
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    print("Experiment:", experiment.name)
    print("Experiment ID:", experiment.experiment_id)
    print("Number of runs:", len(runs))
    if not runs.empty:
        display(runs.head(20))

Experiment: Movie_Recommendation_System
Experiment ID: 1
Number of runs: 2


,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.test_rmse,metrics.validation_mae,metrics.validation_rmse,metrics.baseline_test_rmse,...,params.lr_all,params.n_factors,tags.mlflow.source.name,tags.model_type,tags.acceptance,tags.mlflow.source.type,tags.mlflow.runName,tags.week,tags.mlflow.user,tags.project
0,3f9d72cbf3f64ca7a6b31d1e11cb4880,1,FINISHED,file:///c:/Users/PC/Documents/GitHub/Movie-rec...,2026-11-07 14:03:07.020000+00:00,2026-11-07 14:03:07.401000+00:00,0.855594,0.6684,0.868466,1.039412,...,0.005,50,05_tuning_tracking_serialization.ipynb,Collaborative Filtering,ACCEPTED,NOTEBOOK,SVD_Tuned_Week6,Week 6,PC,Movie Recommendation System
1,bddeb1eeb25f457c9dcb0b256ada1273,1,FINISHED,file:///c:/Users/PC/Documents/GitHub/Movie-rec...,2026-11-07 13:53:06.702000+00:00,2026-11-07 13:53:06.952000+00:00,0.855594,0.6684,0.868466,1.039412,...,0.005,50,05_tuning_tracking_serialization.ipynb,Collaborative Filtering,ACCEPTED,NOTEBOOK,SVD_Tuned_Week6,Week 6,PC,Movie Recommendation System


## 12. Review checklist

In [12]:
review_items = [
    ("Training/test split reproduced with random_state=42", True),
    ("GridSearchCV used training data only", True),
    ("Test set used only for final evaluation", True),
    ("Tuned model evaluated with MAE and RMSE", True),
    ("Acceptance criteria calculated from actual results", True),
    ("Model saved in .pkl format", model_path.exists()),
    ("Saved model reloaded successfully", serialization_ok),
    ("MLflow experiment/run logged", experiment is not None and not runs.empty)
]
review_df = pd.DataFrame([
    {"review_item": item, "passed": bool(ok)} for item, ok in review_items
])
display(review_df)
review_df.to_csv(PROCESSED_DIR / "week6_review_checklist.csv", index=False)
print("Checklist:", "ALL PASSED" if review_df["passed"].all() else "SOME ITEMS NEED ATTENTION")

,review_item,passed
0,Training/test split reproduced with random_sta...,True
1,GridSearchCV used training data only,True
2,Test set used only for final evaluation,True
3,Tuned model evaluated with MAE and RMSE,True
4,Acceptance criteria calculated from actual res...,True
5,Model saved in .pkl format,True
6,Saved model reloaded successfully,True
7,MLflow experiment/run logged,True


Checklist: ALL PASSED


## 13. Weekly Status Report No. 4

In [13]:
status_report_4 = f'''
WEEKLY STATUS REPORT NO. 4 – TUNING, TRACKING AND SERIALIZATION
Project: Movie Recommendation System
Reporting week: Week 6

Completed work:
- Tuned SVD hyperparameters using 3-fold cross-validation on training data only.
- Trained the selected SVD model on the complete training partition.
- Evaluated the tuned model on the untouched test partition.
- Logged parameters, validation metrics and test metrics to MLflow.
- Serialized the model using joblib and verified that it can be reloaded.
- Executed the acceptance criteria and review checklist.

Best parameters: {best_params}
Cross-validation RMSE: {best_cv_rmse:.4f}
Cross-validation MAE: {best_cv_mae:.4f}
Held-out test RMSE: {tuned_test_rmse:.4f}
Held-out test MAE: {tuned_test_mae:.4f}
Week 5 baseline test RMSE: {baseline_test_rmse:.4f}
Week 5 baseline test MAE: {baseline_test_mae:.4f}

Acceptance result: {"ACCEPTED" if accepted and serialization_ok else "NOT ACCEPTED"}
Serialization verification: {"PASS" if serialization_ok else "FAIL"}

Risks/issues:
- If the tuned model does not outperform the baseline, further tuning or model review is required.
- MLflow UI must be opened locally to inspect the experiment dashboard.

Next steps:
- Complete deployment and KPI dashboard work in Week 7.
- Prepare final project documentation and defense.
'''
(REPORT_DIR / "status_report_no_4.txt").write_text(status_report_4, encoding="utf-8")
print(status_report_4)


WEEKLY STATUS REPORT NO. 4 – TUNING, TRACKING AND SERIALIZATION
Project: Movie Recommendation System
Reporting week: Week 6

Completed work:
- Tuned SVD hyperparameters using 3-fold cross-validation on training data only.
- Trained the selected SVD model on the complete training partition.
- Evaluated the tuned model on the untouched test partition.
- Logged parameters, validation metrics and test metrics to MLflow.
- Serialized the model using joblib and verified that it can be reloaded.
- Executed the acceptance criteria and review checklist.

Best parameters: {'n_factors': 50, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.05, 'random_state': 42}
Cross-validation RMSE: 0.8685
Cross-validation MAE: 0.6684
Held-out test RMSE: 0.8556
Held-out test MAE: 0.6619
Week 5 baseline test RMSE: 1.0394
Week 5 baseline test MAE: 0.8340

Acceptance result: ACCEPTED
Serialization verification: PASS

Risks/issues:
- If the tuned model does not outperform the baseline, further tuning or model review 

## 14. Open the MLflow dashboard

In [14]:
# Run this in a terminal from the project root (not as a Python statement):
# mlflow ui
# Then open the local URL printed in the terminal, usually http://127.0.0.1:5000
print("To view the MLflow dashboard, run `mlflow ui` in a terminal from the project root.")

To view the MLflow dashboard, run `mlflow ui` in a terminal from the project root.
